# Tính tổng các chữ số viết tay bằng CNN + LSTM (PyTorch)

Bản PyTorch của `Yêu cầu/examples/3. CNN + LSTM/MNIST_CNN_LSTM.ipynb`, giữ nguyên thứ tự mục và cấu hình của gốc.

- Dữ liệu tự sinh từ MNIST: mỗi mẫu là chuỗi 8 ảnh 28×28, trong đó 1–8 ảnh đầu là chữ số thật, phần còn lại là ảnh 0; nhãn là tổng các chữ số (0–72). Bài toán hồi quy.
- Mô hình: mỗi ảnh trong chuỗi đi qua cùng một CNN (3 khối Conv 64/128/256 same + MaxPool + Dropout 0.2, bọc `TimeDistributed`), duỗi thành vector 2304 chiều, rồi qua LSTM 128 (trả cả chuỗi) → LSTM 128 (trả bước cuối) → Dropout 0.22 → Dense 1.
- Huấn luyện: `mean_absolute_error`, SGD lr 0.01, batch 128, 30 epoch; validation là tập test như gốc.

| | Keras gốc (Keras 2.0.9, GTX 1080) | PyTorch |
|---|---|---|
| Số tham số | 1.747.073 | 1.748.097 (+1.024, giải thích ở mục (2)) |
| MAE trên test | 0.6502 | xem mục (4) |
| Thời gian | 26,6 phút | xem mục (3) |

# Import libraries

Keras gốc:
```python
import keras
from keras.datasets import mnist
from keras.models import Model
from keras.layers import Input, TimeDistributed, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, LSTM
from keras.optimizers import SGD
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # để import pytorch/common.py
from common import *

import contextlib, io
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

# True: chỉ chạy thử (ít dữ liệu, 1 epoch) để kiểm code trên CPU; bản nộp phải là False (chạy GPU)
CHAY_THU = False

set_seed(0)
device = get_device()
TEN = "MNIST_CNN_LSTM"

# (1) Data

Keras gốc:
```python
(X_train_raw, y_train_temp), (X_test_raw, y_test_temp) = mnist.load_data()
X_train_raw = X_train_raw.astype('float32')
X_test_raw  = X_test_raw.astype('float32')
X_train_raw /= 255
X_test_raw  /= 255
```
`torchvision` cho cùng 60.000 / 10.000 ảnh MNIST, cùng thứ tự với `keras.datasets.mnist`.

In [ ]:
with contextlib.redirect_stderr(io.StringIO()):   # ẩn thanh tiến trình khi tải lần đầu
    mnist_train = torchvision.datasets.MNIST(DATA_DIR / "mnist", train=True, download=True)
    mnist_test = torchvision.datasets.MNIST(DATA_DIR / "mnist", train=False, download=True)

X_train_raw = mnist_train.data.numpy().astype('float32') / 255   # (60000, 28, 28)
X_test_raw = mnist_test.data.numpy().astype('float32') / 255
y_train_temp = mnist_train.targets.numpy()
y_test_temp = mnist_test.targets.numpy()

print('X_train_raw shape:', X_train_raw.shape)
print(X_train_raw.shape[0], 'train samples')
print(X_test_raw.shape[0], 'test samples')

## Sinh dữ liệu mới
- Mỗi mẫu: chuỗi `maxToAdd = 8` ảnh, `numToAdd` (1–8, ngẫu nhiên) ảnh đầu là chữ số MNIST chọn ngẫu nhiên, còn lại là ảnh 0.
- Nhãn: tổng các chữ số trong chuỗi.

Ghi chú markdown của gốc nói ảnh 48×48, nhưng code thật dùng `size = 28`.

Giữ **đúng code numpy và `np.random.seed(2018)`** của gốc, nên chuỗi số ngẫu nhiên giống hệt và dữ liệu sinh ra trùng với bản Keras. Hai chỗ khác:
- Mảng tạo kiểu float32 thay float64 (gốc `np.zeros` mặc định float64 tốn 3 GB cho tập train); giá trị không đổi vì ảnh gốc đã là float32.
- Đặt kênh trước chiều ảnh: `(N, 8, 1, 28, 28)` thay `(N, 8, 28, 28, 1)`.

In [ ]:
n_train = 2000 if CHAY_THU else X_train_raw.shape[0]
n_test = 500 if CHAY_THU else X_test_raw.shape[0]

np.random.seed(2018)
maxToAdd = 8
size = 28


def sinh_du_lieu(X_raw, y_raw, n):
    """Đúng vòng lặp của gốc (cùng thứ tự gọi np.random), chỉ đổi shape và kiểu."""
    x = np.zeros((n, maxToAdd, 1, size, size), dtype=np.float32)
    y = []
    for i in range(0, n):
        # decide how many MNIST images to put in that tensor
        numToAdd = int(np.ceil(np.random.rand() * maxToAdd))
        # sample that many images
        indices = np.random.choice(X_raw.shape[0], size=numToAdd)
        x[i, 0:numToAdd, 0, :, :] = X_raw[indices]
        # sum up the outputs for new output
        y.append(np.sum(y_raw[indices]))
    return x, np.array(y)


x_train, y_train = sinh_du_lieu(X_train_raw, y_train_temp, n_train)
print("x_train shape: ", x_train.shape)
print("y_train shape: ", y_train.shape)

x_test, y_test = sinh_du_lieu(X_test_raw, y_test_temp, n_test)
print("x_test shape: ", x_test.shape)
print("y_test shape: ", y_test.shape)
print("5 nhãn đầu của tập train:", y_train[:5], "| tổng lớn nhất:", y_train.max())

# (2) Declare model

Keras gốc:
```python
input_image = Input(shape=(maxToAdd, 28, 28, 1), name='Input')
x = TimeDistributed(Conv2D(filters=64, kernel_size=(3, 3), padding='same', activation='relu'), name='conv1')(input_image)
x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)), name='pool1')(x)
x = TimeDistributed(Dropout(rate=0.2), name='conv_dropout1')(x)
x = TimeDistributed(Conv2D(filters=128, kernel_size=(3, 3), padding='same', activation='relu'), name='conv2')(x)
x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)), name='pool2')(x)
x = TimeDistributed(Dropout(rate=0.2), name='conv_dropout2')(x)
x = TimeDistributed(Conv2D(filters=256, kernel_size=(3, 3), padding='same', activation='relu'), name='conv3')(x)
x = TimeDistributed(MaxPooling2D(pool_size=(2, 2)), name='pool3')(x)
x = TimeDistributed(Dropout(rate=0.2), name='conv_dropout3')(x)
x = TimeDistributed(Flatten(), name='flatten')(x)
x = LSTM(units=128, return_sequences=True, name='lstm1')(x)
x = LSTM(units=128, return_sequences=False, name='lstm2')(x)
x = Dropout(rate=0.22, name='lstm_dropout')(x)
output_label = Dense(units=1, name='fc3_10ways_softmax')(x)
model = Model(inputs=input_image, outputs=output_label, name='mnist_mlp')
```

Khác Keras:
- **`TimeDistributed`**: PyTorch không có lớp này. Gộp chiều thời gian vào chiều batch `(N, 8, 1, 28, 28)` → `(N·8, 1, 28, 28)`, cho qua CNN, rồi tách lại `(N, 8, 2304)`. CNN dùng chung trọng số cho 8 ảnh, đúng như `TimeDistributed`.
- **Hai lớp LSTM**: để hai `nn.LSTM` riêng tên `lstm1`, `lstm2` cho khớp Keras. `lstm1` trả cả chuỗi (`return_sequences=True`); `lstm2` chỉ lấy bước cuối `out[:, -1]` (`return_sequences=False`). `batch_first=True` để tensor có dạng `(N, T, đặc trưng)` như Keras.
- **Số tham số lệch +1.024**: mỗi cổng LSTM của PyTorch có hai vector bias (`bias_ih`, `bias_hh`), Keras có một. Hai lớp × 4 cổng × 128 = 1.024, nên 1.747.073 + 1.024 = 1.748.097. Tổng hai bias của PyTorch tương đương một bias của Keras, nên khả năng biểu diễn như nhau.
- **Hàm kích hoạt cổng**: LSTM Keras 2.0.9 mặc định `recurrent_activation='hard_sigmoid'` (xấp xỉ tuyến tính từng đoạn của sigmoid); `nn.LSTM` dùng sigmoid thật và không đổi được. Đây là khác biệt duy nhất về kiến trúc không giữ được.
- Khởi tạo LSTM như Keras (`init_keras_like`): kernel glorot uniform, kernel hồi quy orthogonal, bias 0, bias cổng quên 1.
- Đầu ra hồi quy nên không có softmax (tên lớp `fc3_10ways_softmax` giữ theo gốc dù không có softmax).

In [ ]:
class CnnLstm(nn.Module):
    """Tên gốc 'mnist_mlp'; các lớp trùng `name=` Keras."""

    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.conv_dropout1 = nn.Dropout(0.2)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2)
        self.conv_dropout2 = nn.Dropout(0.2)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.pool3 = nn.MaxPool2d(2)
        self.conv_dropout3 = nn.Dropout(0.2)
        self.flatten = nn.Flatten()
        self.lstm1 = nn.LSTM(256 * 3 * 3, 128, batch_first=True)
        self.lstm2 = nn.LSTM(128, 128, batch_first=True)
        self.lstm_dropout = nn.Dropout(0.22)
        self.fc3_10ways_softmax = nn.Linear(128, 1)

    def forward(self, x):                                       # (N, 8, 1, 28, 28)
        n, t = x.shape[:2]
        x = x.reshape(n * t, *x.shape[2:])                      # (N·8, 1, 28, 28): TimeDistributed
        x = self.conv_dropout1(self.pool1(F.relu(self.conv1(x))))   # (N·8, 64, 14, 14)
        x = self.conv_dropout2(self.pool2(F.relu(self.conv2(x))))   # (N·8, 128, 7, 7)
        x = self.conv_dropout3(self.pool3(F.relu(self.conv3(x))))   # (N·8, 256, 3, 3)
        x = self.flatten(x).reshape(n, t, -1)                   # (N, 8, 2304)
        x, _ = self.lstm1(x)                                    # (N, 8, 128): return_sequences=True
        x, _ = self.lstm2(x)                                    # (N, 8, 128)
        x = self.lstm_dropout(x[:, -1])                         # (N, 128): bước cuối
        return self.fc3_10ways_softmax(x).squeeze(1)            # (N,): tổng ước lượng


model = init_keras_like(CnnLstm()).to(device)
summary(model, input_size=(1, maxToAdd, 1, 28, 28))

In [ ]:
n_params = count_params(model)
print("Số tham số PyTorch:", n_params, "| Keras: 1747073 | lệch:", n_params - 1_747_073)
assert n_params == 1_747_073 + 2 * 4 * 128     # = 1.748.097, lệch do bias kép của LSTM PyTorch

# (3) Train defined model

Keras gốc:
```python
loss = 'mean_absolute_error'
lr = 0.01
model.compile(loss=loss, optimizer=SGD(lr=lr))
batch_size = 128
epochs = 30
history = model.fit(x_train, y_train, validation_data=(x_test, y_test),
                    batch_size=batch_size, epochs=epochs)
```
Gốc không khai báo `metrics`, nên log chỉ có `loss` (MAE) và `val_loss`. Nhãn là số thực nên đổi sang float32.

In [ ]:
loss_fn = nn.L1Loss()           # = 'mean_absolute_error'
lr = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

batch_size = 128
epochs = 1 if CHAY_THU else 30

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)
train_loader = make_loader(torch.from_numpy(x_train), y_train_t, batch_size=batch_size, shuffle=True)
test_loader = make_loader(torch.from_numpy(x_test), y_test_t, batch_size=batch_size, shuffle=False)

starting_time = time.time()
history = fit(model, train_loader, test_loader, loss_fn, optimizer, epochs)
train_time = time.time() - starting_time
print('> training time is %.4f minutes' % (train_time / 60))

### Training time
- Gốc: GTX 1080 khoảng 26,6 phút, Core i7 khoảng 808 phút (~13,5 giờ) cho 30 epoch.
- Bản PyTorch: xem dòng `training time` ở trên.

In [ ]:
fig = plot_history(history, None if CHAY_THU else TEN, title="MNIST CNN + LSTM (PyTorch), loss = MAE")
plt.show()

# (4) Evaluate trained model

Kết quả gốc: `Mean absolute error of testing set: 0.6502`.

In [ ]:
score, _ = evaluate(model, test_loader, loss_fn)
print('Mean absolute error of testing set:', score)

In [ ]:
if not CHAY_THU:
    save_history(history, TEN)
    save_metrics(TEN, test_loss=score, test_metric=score, n_params=n_params,
                 epochs_ran=len(history["loss"]), best_epoch=None, train_time_s=train_time,
                 device=device_name(device), seed=0, data_seed=2018)
    ghi_so_sanh(TEN, "3. CNN + LSTM", "mae", keras=0.6502, pytorch=score,
                keras_params=1747073, pytorch_params=n_params, epochs_keras=30, epochs_pytorch=epochs,
                optimizer="SGD", lr=lr, batch=batch_size, device=device_name(device),
                train_time_s=train_time, seed=0,
                ghi_chu="LSTM PyTorch có 2 bias → +1024 tham số; cổng dùng sigmoid thay hard_sigmoid; dữ liệu sinh với seed 2018 như gốc")

## Dự đoán một mẫu test ngẫu nhiên

In [ ]:
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]

plt.figure(figsize=(10, 5))
for indx in range(maxToAdd):
    plt.subplot(1, maxToAdd, indx + 1)
    test_img = x_test[random_test_index, indx, 0, :, :]
    plt.axis('off')
    plt.imshow(test_img, cmap='gray')

# estimate summation of testing images
est_sum = float(predict(model, torch.from_numpy(x_test[random_test_index:random_test_index + 1]))[0])

print('> actual summation = %d' % y_test[random_test_index])
print('> estimated summation = %.3f' % est_sum)
plt.show()